# 03 - Tuning → Training → Prediction

End-to-End Pipeline:
1. **Daten laden** (oder fetchen falls nicht vorhanden)
2. **Optuna Hyperparameter-Tuning** (jeder Trial → DagsHub MLflow)
3. **Finales Training** mit besten Parametern
4. **Model Registry** — nur registriert wenn besser als aktuelles
5. **Prediction** — Modell aus Registry laden & vorhersagen

In [1]:
import mlflow
from sklearn.metrics import accuracy_score, f1_score

from backend.core.config import settings
from backend.infra.dagshub_init import init_dagshub
from backend.infra.database import get_data_store
from backend.ml.training.extra_tree import ExtraTreesTrainer
from backend.ml.training.tuning.optuna_tuner import tune
from backend.ml.registry.model_registry import log_and_register
from backend.core.prediction.mlflow_predict import MLflowPredictor

init_dagshub()
print(f"DagsHub:  {settings.dagshub.repo_full}")
print(f"Tracking: {mlflow.get_tracking_uri()}")

/Users/e.autenrieth/Documents/GitHub/ml-stock-prediction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as erikautenrieth

Initialized MLflow to track repo "erikautenrieth/stock-prediction-hub"

Repository erikautenrieth/stock-prediction-hub initialized!

2026-03-08 20:12:50 [info     ] dagshub_initialized            experiment=sp500_prediction repo=erikautenrieth/stock-prediction-hub tracking_uri=https://dagshub.com/erikautenrieth/stock-prediction-hub.mlflow
DagsHub:  erikautenrieth/stock-prediction-hub
Tracking: https://dagshub.com/erikautenrieth/stock-prediction-hub.mlflow


## 1. Features laden

In [2]:
store = get_data_store()

try:
    df = store.load_features()
    print(f"Features: {df.shape[0]} rows, {df.shape[1]} cols")
    print(f"Target: {df['Target'].value_counts().to_dict()}")
except FileNotFoundError:
    print("Keine Features → fetche Daten...")
    from backend.workflows.fetch_data import fetch_and_store
    df, manifest = fetch_and_store()
    print(f"Done: {manifest.row_count} rows, {len(manifest.columns)} cols")

2026-03-08 20:13:05 [info     ] dagshub_storage_initialized    bucket=stock-prediction-hub
2026-03-08 20:13:05 [info     ] data_store_remote_enabled     
Features: 6403 rows, 53 cols
Target: {1: 3854, 0: 2549}


## 2. Train/Test Split

In [ ]:
trainer = ExtraTreesTrainer()
x_train, x_test, y_train, y_test = trainer.prepare(df)

print(f"Train: {x_train.shape}  (Up: {sum(y_train==1)}, Down: {sum(y_train==0)})")
print(f"Test:  {x_test.shape}  (Up: {sum(y_test==1)}, Down: {sum(y_test==0)})")

## 3. Optuna Tuning → Training → Model Registry

In [ ]:
N_TRIALS = 50

with mlflow.start_run(run_name=f"train_{trainer.name()}") as run:
    # --- Optuna Tuning ---
    print(f"Optuna tuning ({N_TRIALS} trials)...")
    best_params = tune(trainer, x_train, y_train, x_test, y_test, n_trials=N_TRIALS)

    print(f"\nBeste Parameter:")
    for k, v in best_params.items():
        print(f"  {k}: {v}")

    # --- Finales Training ---
    model = trainer.build(best_params)
    model.fit(x_train, y_train)

    preds = model.predict(x_test)
    metrics = {
        "accuracy": accuracy_score(y_test, preds),
        "f1": f1_score(y_test, preds, average="weighted"),
    }
    print(f"\nAccuracy: {metrics['accuracy']:.4f}")
    print(f"F1:       {metrics['f1']:.4f}")

    # --- Model Registry (nur wenn besser) ---
    model_uri = log_and_register(
        model=model,
        x_train=x_train,
        y_train=y_train,
        params=best_params,
        metrics=metrics,
    )
    print(f"\nModel URI: {model_uri}")
    print(f"Run ID:    {run.info.run_id}")
    print(f"DagsHub:   {settings.dagshub.tracking_uri}")

## 4. Prediction — Modell aus Registry laden

In [ ]:
predictor = MLflowPredictor()
predictor.load_model()

last_row = df.drop("Target", axis=1).tail(1)
pred = predictor.predict(last_row)[0]
proba = predictor.predict_proba(last_row)[0]

direction = "UP ↑" if pred == 1 else "DOWN ↓"
print(f"Prediction: {direction}")
print(f"Confidence: {max(proba):.2%}")
print(f"Horizon:    {settings.stock.prediction_horizon_days} Tage")
print(f"Date:       {last_row.index[0]}")

## 5. Test auf gesamtem Test-Set

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

test_features = pd.DataFrame(x_test, columns=trainer._feature_names)
test_preds = predictor.predict(test_features)
test_proba = predictor.predict_proba(test_features)

print(classification_report(y_test, test_preds, target_names=["DOWN", "UP"]))
print(f"Mean confidence: {test_proba.max(axis=1).mean():.2%}")